In [ ]:
!pip install -q -U sentence-transformers

In [ ]:
# Векторы для итоговых корпусов

import os
import re
import json
import shutil
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from huggingface_hub import model_info
from google.colab import files

MODEL = "sergeyzh/LaBSE-ru-turbo"
PROMPT = "categorize: "
CORPORA = ["yt", "tg", "rr"]
VERSIONS = ["orig", "level1", "level2", "level3"]
OUT = "/content/embeddings"

MARKUP = re.compile("|".join([
    r"[\w.+-]+@[\w-]+\.[A-Za-z]+",                                              # почта
    r"https?://\S*",                                                            # ссылки
    r"\bwww\.\S+",
    r"\b[\w.-]+\.(?:com|ru|org|net|io|dev|ai|me|su|info|tv|gg|app|рф)\b\S*",    # домены с путем
    r"@\w+",                                                                    # упоминания и заглушки @user_...
    r"(?:~|\.)?(?:/[A-Za-z_][\w.-]*){2,}/?",                                    # пути unix
    r"\b[A-Za-z]:\\\S+"                                                         # пути windows
]))


def strip_markup(text):
    return MARKUP.sub(" ", str(text))


assert torch.cuda.is_available(), "включить GPU"
model = SentenceTransformer(MODEL, device="cuda")
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT)

for corpus in CORPORA:
    df = pd.read_parquet(f"/content/{corpus}.parquet")
    for version in VERSIONS:
        vectors = model.encode(df[version].map(strip_markup).tolist(), prompt=PROMPT, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
        out = pd.DataFrame(vectors.astype(np.float32), columns=[f"dim_{i}" for i in range(vectors.shape[1])])
        out.insert(0, "uid", df.uid.values)
        out.to_parquet(f"{OUT}/{corpus}_{version}.parquet", index=False)
        print(f"{corpus} {version}: {out.shape}")

with open(f"{OUT}/manifest.json", "w") as file:
    json.dump({"model": MODEL, "revision": model_info(MODEL).sha, "prompt": PROMPT, "normalized": True, "strip_markup": True}, file, indent=2)

shutil.make_archive("/content/embeddings", "zip", OUT)
files.download("/content/embeddings.zip")